In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("Lakehouse - Gold Fact & ML") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,io.delta:delta-spark_2.12:3.1.0,mysql:mysql-connector-java:8.0.33") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# 1. Đọc Silver + Dim
df_delivery = spark.read.format("delta").load("s3a://lakehouse/warehouse/delivery_silver")
df_courier = spark.read.format("delta").load("s3a://lakehouse/warehouse/dim_courier_silver")

print(f"Silver: {df_delivery.count()} dòng")

# 2. Ngày giao hàng
df_delivery = df_delivery.withColumn("order_date", F.to_date("accept_ts"))

# 3. Dim chỉ giữ cột SCD2 cần thiết, bỏ trùng để join không nhân dòng
dim = df_courier.select("courier_id", "region_id", "effective_date", "end_date").dropDuplicates()

# 4. Join theo khoảng hiệu lực: effective_date <= order_date < end_date
fact_delivery = df_delivery.alias("d").join(
    dim.alias("c"),
    (F.col("d.courier_id") == F.col("c.courier_id")) &
    (F.col("d.order_date") >= F.col("c.effective_date")) &
    (F.col("d.order_date") < F.col("c.end_date")),
    "inner"
).select(
    "d.order_id", "d.city", "d.courier_id",
    F.col("c.region_id").alias("region_id"),
    "d.aoi_id", "d.order_date",
    "d.distance_km", "d.delivery_minutes"
).dropDuplicates(["order_id"])

fact_count = fact_delivery.count()
print(f"Số dòng bảng Fact: {fact_count}")
if fact_count == 0:
    raise ValueError("Fact rỗng - kiểm tra lại Silver (accept_ts NULL?) và điều kiện join với Dim")

# 5. Ghi Gold
fact_delivery.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("s3a://lakehouse/warehouse/fact_delivery_task_gold")
print("Đã lưu bảng Fact Gold thành công!")

# 6. Train ML dự báo ETA
ml_data = fact_delivery.filter((F.col("distance_km") > 0) & (F.col("delivery_minutes") > 0))
ml_count = ml_data.count()
print(f"Số dòng dùng để train ML: {ml_count}")
if ml_count == 0:
    raise ValueError("ml_data rỗng - kiểm tra distance_km và delivery_minutes")

assembler = VectorAssembler(inputCols=["distance_km"], outputCol="features")
ml_data = assembler.transform(ml_data)

train_data, test_data = ml_data.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(featuresCol="features", labelCol="delivery_minutes")
lr_model = lr.fit(train_data)

predictions = lr_model.transform(test_data)
rmse = RegressionEvaluator(labelCol="delivery_minutes", predictionCol="prediction", metricName="rmse").evaluate(predictions)
r2 = RegressionEvaluator(labelCol="delivery_minutes", predictionCol="prediction", metricName="r2").evaluate(predictions)

print("-" * 50)
print("KẾT QUẢ HUẤN LUYỆN MÔ HÌNH DỰ BÁO ETA")
print(f"- RMSE: {rmse:.2f} phút")
print(f"- R2: {r2:.3f}")
print(f"- Hệ số (Coefficient): {lr_model.coefficients[0]:.2f}")
print(f"- Intercept: {lr_model.intercept:.2f}")
print("-" * 50)

predictions.select("distance_km", "delivery_minutes",
                   F.round("prediction", 2).alias("predicted_minutes")).show(5)

Silver: 201000 dòng
Số dòng bảng Fact: 201000
Đã lưu bảng Fact Gold thành công!
Số dòng dùng để train ML: 201000
--------------------------------------------------
KẾT QUẢ HUẤN LUYỆN MÔ HÌNH DỰ BÁO ETA
- RMSE: 159.86 phút
- R2: 0.086
- Hệ số (Coefficient): 11.48
- Intercept: 179.36
--------------------------------------------------
+------------------+----------------+-----------------+
|       distance_km|delivery_minutes|predicted_minutes|
+------------------+----------------+-----------------+
| 7.404454332724183|           229.0|           264.34|
| 6.175175906895067|           203.0|           250.23|
|1.1578905773621264|            93.0|           192.65|
| 5.123679293401504|           254.0|           238.17|
| 9.748446879328313|           578.0|           291.25|
+------------------+----------------+-----------------+
only showing top 5 rows



In [2]:
dim = spark.read.format("delta").load("s3a://lakehouse/warehouse/dim_courier_silver")
silver = spark.read.format("delta").load("s3a://lakehouse/warehouse/delivery_silver") \
              .withColumn("order_date", F.to_date("accept_ts"))

dim.agg(F.min("effective_date"), F.max("effective_date"), F.countDistinct("courier_id")).show()
silver.agg(F.min("order_date"), F.max("order_date"), F.countDistinct("courier_id")).show()

+-------------------+-------------------+--------------------------+
|min(effective_date)|max(effective_date)|count(DISTINCT courier_id)|
+-------------------+-------------------+--------------------------+
|         2022-05-01|         2022-10-31|                       205|
+-------------------+-------------------+--------------------------+

+---------------+---------------+--------------------------+
|min(order_date)|max(order_date)|count(DISTINCT courier_id)|
+---------------+---------------+--------------------------+
|     2022-05-01|     2022-10-31|                       202|
+---------------+---------------+--------------------------+



In [3]:
fact = spark.read.format("delta").load("s3a://lakehouse/warehouse/fact_delivery_task_gold")
print(fact.stat.corr("distance_km", "delivery_minutes"))
fact.select("delivery_minutes").summary("min", "25%", "50%", "75%", "max").show()

0.2996380604265805
+-------+----------------+
|summary|delivery_minutes|
+-------+----------------+
|    min|             1.0|
|    25%|           113.0|
|    50%|           204.0|
|    75%|           336.0|
|    max|          1440.0|
+-------+----------------+



In [4]:
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import pyspark.sql.functions as F

s = spark.read.format("delta").load("s3a://lakehouse/warehouse/delivery_silver") \
         .withColumn("order_date", F.to_date("accept_ts"))

w = Window.partitionBy("courier_id", "order_date").orderBy("accept_ts", "order_id") \
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)

feat = s \
    .withColumn("accept_hour", F.hour("accept_ts") + F.minute("accept_ts") / 60) \
    .withColumn("dow", F.dayofweek("order_date")) \
    .withColumn("order_rank", F.count("*").over(w)) \
    .withColumn("cum_avg_dist", F.avg("distance_km").over(w)) \
    .withColumn("cum_max_dist", F.max("distance_km").over(w)) \
    .withColumn("region_str", F.col("region_id").cast("string")) \
    .withColumn("label_log", F.log1p("delivery_minutes"))

train = feat.filter(F.col("order_date") < "2022-10-01")
test  = feat.filter(F.col("order_date") >= "2022-10-01")
print(f"train={train.count()}, test={test.count()}")

idx_city   = StringIndexer(inputCol="city", outputCol="city_idx", handleInvalid="keep")
idx_aoi    = StringIndexer(inputCol="aoi_type", outputCol="aoi_type_idx", handleInvalid="keep")
idx_region = StringIndexer(inputCol="region_str", outputCol="region_idx", handleInvalid="keep")

cols = ["distance_km", "accept_hour", "dow", "order_rank",
        "cum_avg_dist", "cum_max_dist", "region_idx", "aoi_type_idx"]
assembler = VectorAssembler(inputCols=cols, outputCol="features", handleInvalid="skip")

gbt = GBTRegressor(featuresCol="features", labelCol="label_log",
                   maxIter=80, maxDepth=6, maxBins=256, seed=42)

model = Pipeline(stages=[idx_city, idx_aoi, idx_region, assembler, gbt]).fit(train)
pred = model.transform(test).withColumn("pred_min", F.expm1("prediction"))

def ev(label, predcol, metric):
    return round(RegressionEvaluator(labelCol=label, predictionCol=predcol, metricName=metric).evaluate(pred), 3)

print("MODEL    rmse", ev("delivery_minutes", "pred_min", "rmse"),
      "| mae", ev("delivery_minutes", "pred_min", "mae"),
      "| r2",  ev("delivery_minutes", "pred_min", "r2"))

med = train.approxQuantile("delivery_minutes", [0.5], 0.01)[0]
mean_train = train.agg(F.avg("delivery_minutes")).first()[0]
base = test.withColumn("p_med", F.lit(med)).withColumn("p_mean", F.lit(mean_train))
for name, c in [("median", "p_med"), ("mean", "p_mean")]:
    print(f"BASELINE {name:6s} rmse",
          round(RegressionEvaluator(labelCol="delivery_minutes", predictionCol=c, metricName="rmse").evaluate(base), 3),
          "| mae",
          round(RegressionEvaluator(labelCol="delivery_minutes", predictionCol=c, metricName="mae").evaluate(base), 3))

for n, v in sorted(zip(cols, model.stages[-1].featureImportances.toArray()), key=lambda x: -x[1]):
    print(f"{n:16s} {v:.3f}")

train=165222, test=35778
MODEL    rmse 140.284 | mae 95.976 | r2 0.281
BASELINE median rmse 169.445 | mae 127.587
BASELINE mean   rmse 165.419 | mae 130.999
distance_km      0.293
region_idx       0.235
accept_hour      0.136
cum_avg_dist     0.115
cum_max_dist     0.106
order_rank       0.053
aoi_type_idx     0.050
dow              0.011


In [5]:
f = spark.read.format("delta").load("s3a://lakehouse/warehouse/delivery_silver") \
         .filter(F.col("delivery_minutes") <= 1440)
print("pearson :", f.stat.corr("distance_km", "delivery_minutes"))
print("log-log :", f.select(F.log1p("distance_km").alias("x"), F.log1p("delivery_minutes").alias("y")).stat.corr("x", "y"))

# Baseline MAE trên tập test (cần train/test từ cell trước)
med = train.approxQuantile("delivery_minutes", [0.5], 0.01)[0]
print("baseline MAE (trung bình):", test.select(F.avg(F.abs(F.col("delivery_minutes") - mean_train))).first()[0])
print("baseline MAE (trung vị)  :", test.select(F.avg(F.abs(F.col("delivery_minutes") - med))).first()[0])

pearson : 0.29963806042658064
log-log : 0.3975257358378113
baseline MAE (trung bình): 130.99868633679836
baseline MAE (trung vị)  : 127.58686902565823


In [6]:
f = spark.read.format("delta").load("s3a://lakehouse/warehouse/delivery_silver") \
         .filter(F.col("delivery_minutes") <= 1440)

f.select("distance_km").summary("min", "25%", "50%", "75%", "95%", "99%", "max").show()

for cap in [10, 20, 50]:
    g = f.filter(F.col("distance_km") <= cap)
    print(f"distance <= {cap} km: n={g.count()}, pearson={g.stat.corr('distance_km', 'delivery_minutes'):.4f}")

+-------+-------------------+
|summary|        distance_km|
+-------+-------------------+
|    min|8.82843415630323E-4|
|    25%|  2.347426981397986|
|    50%|  4.097837656340729|
|    75%|  7.619167498227802|
|    95%| 12.401675073722075|
|    99%| 23.063179727379957|
|    max|  47.73847979335505|
+-------+-------------------+

distance <= 10 km: n=174441, pearson=0.2686
distance <= 20 km: n=197903, pearson=0.2782
distance <= 50 km: n=201000, pearson=0.2996
